# Inference — 12-model pole-inspection pipeline (Colab)

Runs the full chain on an image or folder: **pole → 5 component specialists → NMS → route each to its condition specialist** → `result.csv` + `result.json` + 4 annotated views (`pole / components / conditions / all`).

Weights are auto-discovered by subset name from `MyDrive/finalGo/runs` (where the training notebook saved them). **Runtime → A100/L4 → Run all**, then set `IMAGE` in the last cell.

After this cell: same dep contract as training (ultralytics 8.4+, pillow<11.3, repo `--no-deps` so rfdetr can't corrupt torch/torchvision).

In [ ]:
# @title 1) Repo + deps  (YOLO/inference-only install)
REPO="/content/repo"
!git clone https://github.com/arupa444/trainDronisight.git $REPO 2>/dev/null || (cd $REPO && git pull)
%cd $REPO
!pip -q install uv
# --no-deps: the full project pulls rfdetr, whose torch/torchvision pins corrupt Colab's CUDA
# stack ("torchvision::nms does not exist"). YOLO inference needs none of it.
!uv pip install --system --no-deps -e .
!uv pip install --system "ultralytics>=8.4.60"
!uv pip install --system --reinstall "pillow==11.2.1"   # 11.3+ ImageText/_Ink crashes on Colab py3.12
import torch, torchvision
from torchvision.ops import nms
import PIL; from PIL import ImageDraw, ImageFont
import ultralytics; from ultralytics import YOLO
from inference.pipeline import run_pipeline, discover_weights      # the exact imports inference uses
print(f"torch {torch.__version__} | tv {torchvision.__version__} | Pillow {PIL.__version__} | "
      f"ultralytics {ultralytics.__version__} | CUDA {torch.cuda.is_available()}  -> ALL OK")

In [ ]:
# @title 2) Mount Drive
from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# @title 3) Locate the 12 trained weights on Drive (auto-discovered by subset name)
import os
from shared import config
WEIGHTS_DIR = "/content/drive/MyDrive/finalGo/runs"   # <- where the training notebook saved runs/
os.environ["DRONISIGHT_DATA"] = "/content/data"        # harmless; config paths not needed for inference
from inference.pipeline import discover_weights
found = discover_weights(WEIGHTS_DIR, config.SUBSETS)
print("subset                     weights")
for s in config.SUBSETS:
    print(f"  {'OK ' if s in found else 'MISSING'} {s:24s} {found.get(s,'')}")
assert "pole" in found, f"No pole weights under {WEIGHTS_DIR}"
miss = [s for s in config.SUBSETS if s not in found]
if miss: print("\nNote: missing ->", miss, "(those component/condition types just won't be produced)")

In [ ]:
# @title 4) Run inference on an image OR a folder
# Option A: upload an image
# from google.colab import files; up = files.upload(); IMAGE = "/content/" + next(iter(up))
# Option B: point at a file or folder already on Drive:
IMAGE = "/content/drive/MyDrive/finalGo/test_images"   # <- a file (x.jpg) or a folder of images
OUT   = "/content/drive/MyDrive/finalGo/inference"     # results saved here (per-run subfolder, never overwritten)

# Models were trained on the CLAHE variant -> keep CLAHE on (default). Pads come from config so
# they match how the crop datasets were built. One detector set is built once, reused over all images.
!python -m inference.pipeline --image "$IMAGE" --weights-dir "$WEIGHTS_DIR" --out-dir "$OUT"

In [ ]:
# @title 5) Preview the annotated 'all' views from the latest run
import glob, os
from IPython.display import Image, display
runs = sorted(glob.glob(f"{OUT}/*_inference*"), key=os.path.getmtime)
assert runs, f"no runs under {OUT}"
latest = runs[-1]; print("latest run:", latest)
print("CSV:", os.path.join(latest, "result.csv"))
for p in sorted(glob.glob(f"{latest}/viz/all/*.jpg"))[:8]:
    print(os.path.basename(p)); display(Image(filename=p, width=900))